# Problem 4(b): Chest Clinic Network

Skeleton for computing $p(d)$, $p(d\mid s=\text{tr})$, $p(d\mid s=\text{fa})$
from the Asia / Chest-Clinic Bayesian network.

Variables: `a` (visit to Asia), `t` (tuberculosis), `s` (smoker), `l` (lung cancer),
`b` (bronchitis), `e` (either t or l), `x` (positive x-ray), `d` (dyspnea).
Each variable is binary: `'tr'` or `'fa'`.

Graph structure (from the assignment figure):
`a -> t`, `s -> l`, `s -> b`, `t -> e`, `l -> e`, `e -> x`, `e -> d`, `b -> d`.

So the joint factorizes as (chain rule along the graph):

$$p(a,t,s,l,b,e,x,d) = p(a)\,p(t\mid a)\,p(s)\,p(l\mid s)\,p(b\mid s)\,p(e\mid t,l)\,p(x\mid e)\,p(d\mid e,b)$$

In [7]:
import itertools

VALUES = ['tr', 'fa']

## Worked mini-example (not the real problem -- just a template to copy)

Before tackling the real 8-variable network, here's the exact same pattern
on a tiny toy network with only 2 variables, fully filled in and run, so you
can see what "correct" looks like before you do it for real below.

Toy network: `X -> Y` (X has no parents, Y's only parent is X). Made-up CPTs:
`p(X=tr) = 0.6`, `p(Y=tr | X=tr) = 0.9`, `p(Y=tr | X=fa) = 0.2`.

This cell is complete and runnable -- run it, look at the output, then use
the exact same 3-step pattern (fill CPTs -> write joint_prob -> sum out
variables you don't want) on the real problem in the cells below.

In [8]:
# --- Step 1: CPTs, fully filled in (this is what your real CPTs should look like) ---
p_x_toy = {'tr': 0.6, 'fa': 0.4}
p_y_given_x_toy = {
    ('tr',): {'tr': 0.9, 'fa': 0.1},
    ('fa',): {'tr': 0.2, 'fa': 0.8},
}

# --- Step 2: joint_prob, fully filled in ---
def joint_prob_toy(x, y):
    """Returns p(x, y) = p(x) * p(y|x)."""
    prob = 1.0
    prob *= p_x_toy[x]
    prob *= p_y_given_x_toy[(x,)][y]
    return prob

print('joint_prob_toy(tr, tr) =', joint_prob_toy('tr', 'tr'))  # should be 0.6*0.9 = 0.54
print('joint_prob_toy(fa, tr) =', joint_prob_toy('fa', 'tr'))  # should be 0.4*0.2 = 0.08

# --- Step 3: marginalize out X to get p(Y), fully filled in ---
def marginal_y_toy():
    p_y = {'tr': 0.0, 'fa': 0.0}
    for x, y in itertools.product(VALUES, repeat=2):
        p_y[y] += joint_prob_toy(x, y)
    return p_y

print('p(Y):', marginal_y_toy())
# Sanity check by hand: p(Y=tr) = p(X=tr)p(Y=tr|X=tr) + p(X=fa)p(Y=tr|X=fa)
#                               = 0.6*0.9 + 0.4*0.2 = 0.54 + 0.08 = 0.62
# -- confirm the printed p_y['tr'] matches 0.62.

joint_prob_toy(tr, tr) = 0.54
joint_prob_toy(fa, tr) = 0.08000000000000002
p(Y): {'tr': 0.6200000000000001, 'fa': 0.38000000000000006}


## Step 1: Fill in the CPTs from the table given in the assignment

Each CPT below is a dict. For a variable with no parents, key by its own value.
For a variable with parents, key by a tuple of parent values (in the order shown),
then the variable's own value.

TODO: replace every `None` with the number from the assignment's table.

In [9]:
# Clue: read every value straight off the table given in the assignment PDF
# (Problem 4b). Each line below corresponds to exactly one entry in that
# table -- e.g. p_a['tr'] = p(a=tr), and p_t_given_a[('tr',)]['tr'] = p(t=tr|a=tr).
# Remember every CPT's two entries for a fixed parent config must sum to 1,
# so p_a['fa'] = 1 - p_a['tr'], p_t_given_a[('tr',)]['fa'] = 1 - p_t_given_a[('tr',)]['tr'], etc.
# -- you only need to type in the 'tr' numbers from the table and derive 'fa' = 1 - 'tr'.

# p(a): no parents
p_a = {
    'tr': 0.01,  # p(a=tr)
    'fa': 0.99,  # = 1 - p(a=tr)
}

# p(s): no parents
p_s = {
    'tr': 0.5,  # p(s=tr)
    'fa': 0.5,  # = 1 - p(s=tr)
}

# p(t | a): parent is a
p_t_given_a = {
    ('tr',): {'tr': 0.05, 'fa': 0.95},  # a=tr -> p(t=tr|a=tr), p(t=fa|a=tr)
    ('fa',): {'tr': 0.01, 'fa': 0.99},  # a=fa -> p(t=tr|a=fa), p(t=fa|a=fa)
}

# p(l | s): parent is s
p_l_given_s = {
    ('tr',): {'tr': 0.1, 'fa': 0.9},
    ('fa',): {'tr': 0.01, 'fa': 0.99},
}

# p(b | s): parent is s
p_b_given_s = {
    ('tr',): {'tr': 0.6, 'fa': 0.4},
    ('fa',): {'tr': 0.3, 'fa': 0.7},
}

# p(x | e): parent is e
p_x_given_e = {
    ('tr',): {'tr': 0.98, 'fa': 0.02},
    ('fa',): {'tr': 0.05, 'fa': 0.95},
}

# p(d | e, b): parents are e, b (in that order)
p_d_given_eb = {
    ('tr', 'tr'): {'tr': 0.9, 'fa': 0.1},
    ('tr', 'fa'): {'tr': 0.7, 'fa': 0.3},
    ('fa', 'tr'): {'tr': 0.8, 'fa': 0.2},
    ('fa', 'fa'): {'tr': 0.1, 'fa': 0.9},
}

# p(e | t, l): deterministic rule given in the assignment --
# p(e=tr | t, l) = 0 only if BOTH t and l are 'fa', otherwise 1.
# Clue: there are 4 (t, l) combinations. Only the ('fa', 'fa') row has
# p(e=tr)=0 (so p(e=fa)=1); the other 3 rows all have p(e=tr)=1 (so p(e=fa)=0).
# TODO: fill this in following that rule (no need to guess -- it's fully specified).
p_e_given_tl = {
    ('tr', 'tr'): {'tr': 1, 'fa': 0},
    ('tr', 'fa'): {'tr': 1, 'fa': 0},
    ('fa', 'tr'): {'tr': 1, 'fa': 0},
    ('fa', 'fa'): {'tr': 0, 'fa': 1},
}

## Step 2: Joint probability of one full assignment

TODO: complete `joint_prob` by multiplying the 8 CPT factors together,
using the factorization written in the markdown cell above. Each CPT
lookup follows the pattern `p_t_given_a[(a,)][t]` for single-parent CPTs
or `p_d_given_eb[(e, b)][d]` for two-parent CPTs, and `p_a[a]` for no-parent CPTs.

In [10]:
def joint_prob(a, t, s, l, b, e, x, d):
    """Returns p(a,t,s,l,b,e,x,d) using the chain-rule factorization."""
    # Clue: just uncomment each line below and multiply it into `prob`.
    # Each lookup pulls the ONE number matching the current assignment out
    # of the dict you filled in above -- e.g. p_t_given_a[(a,)][t] means
    # "look up the row for parent value a, then the column for child value t".
    prob = 1.0
    prob *= p_a[a]
    prob *= p_t_given_a[(a,)][t]
    prob *= p_s[s]
    prob *= p_l_given_s[(s,)][l]
    prob *= p_b_given_s[(s,)][b]
    prob *= p_e_given_tl[(t, l)][e]
    prob *= p_x_given_e[(e,)][x]
    prob *= p_d_given_eb[(e, b)][d]
    return prob

## Step 3: Marginalize to get p(d)

To get $p(d)$, sum `joint_prob(...)` over every possible value of the
other 7 variables, for each fixed value of `d`. `itertools.product` is
handy for enumerating all combinations.

TODO: complete the loop below.

In [11]:
def marginal_d():
    p_d = {'tr': 0.0, 'fa': 0.0}
    for a, t, s, l, b, e, x, d in itertools.product(VALUES, repeat=8):
        # Clue: this loop already binds `d` to the current variable's value
        # (either 'tr' or 'fa'). You just need to add this one assignment's
        # joint probability into the bucket for that value of d:
        p_d[d] += joint_prob(a, t, s, l, b, e, x, d)
        pass
    return p_d

p_d = marginal_d()
print('p(d):', p_d)
# sanity check: p_d['tr'] + p_d['fa'] should equal 1.0

p(d): {'tr': 0.4359706, 'fa': 0.5640293999999999}


## Step 4: Condition on s to get p(d | s=tr) and p(d | s=fa)

To condition on evidence, restrict the sum to assignments where `s`
equals the fixed value, then **normalize** the resulting two numbers
(for d='tr' and d='fa') so they sum to 1 -- that normalization is what
divides out $p(s=\text{tr})$ or $p(s=\text{fa})$ for you automatically.

TODO: complete `conditional_d_given_s`.

In [12]:
def conditional_d_given_s(s_value):
    unnormalized = {'tr': 0.0, 'fa': 0.0}
    for a, t, s, l, b, e, x, d in itertools.product(VALUES, repeat=8):
        if s != s_value:
            continue
        # Clue: same accumulation as marginal_d above, just restricted to
        # the assignments that survived the `if s != s_value: continue` filter:
        unnormalized[d] += joint_prob(a, t, s, l, b, e, x, d)
        pass
    total = unnormalized['tr'] + unnormalized['fa']
    return {k: v / total for k, v in unnormalized.items()}

print('p(d | s=tr):', conditional_d_given_s('tr'))
print('p(d | s=fa):', conditional_d_given_s('fa'))

p(d | s=tr): {'tr': 0.552808, 'fa': 0.44719200000000003}
p(d | s=fa): {'tr': 0.31913319999999995, 'fa': 0.6808668}
